# Problem 019:  Counting Sundays

> You are given the following information, but you may prefer to do some research for yourself.
>
>>    1 Jan 1900 was a Monday. <br>
>>    Thirty days has September, <br>
>>    April, June and November. <br>
>>    All the rest have thirty-one, <br>
>>    Saving February alone, <br>
>>    Which has twenty-eight, rain or shine. <br>
>>    And on leap years, twenty-nine. <br>
>>    A leap year occurs on any year evenly divisible by 4, but not on a century unless it is divisible by 400.
>
> How many Sundays fell on the first of the month during the twentieth century (1 Jan 1901 to 31 Dec 2000)?


## Using `datatime`

The python package `datetime` will readily give you the day of the week of any date.

The `year` variables are the numerical year, the `month` variables are between $1-12$, and the `day` variables are between $0-6$, $0$ being Monday.

In [3]:
%%time

from datetime import date

def p019_python(year_a: int, month_a: int, year_b: int, month_b: int, day: int) -> int:
    val = 0

    if (year_a, month_a) > (year_b, month_b):
        return 0

    year = year_a
    month = month_a
    while year_b != year or month_b != month:
        if date(year, month, 1).weekday() == day:
            val += 1
        if month == 12:
            month = 1
            year += 1
        else:
            month += 1

    if date(year, month, 1).weekday() == day:
        val += 1

    return val
    
year_a = 1901
month_a = 1
year_b = 2000
month_b = 12
day = 6

ans = p019_python(year_a, month_a, year_b, month_b, day)
print(ans)


171
CPU times: user 1.62 ms, sys: 25 μs, total: 1.65 ms
Wall time: 1.41 ms


## Non-Pythonic Solution

There are a few ways to approach this one without using `datetime`.  One would be to build a helper function that takes a date and returns the day of the week.  Whereas that would be a useful piece of code, it would be inefficient to use here since it would require starting on 1 Jan 1900 each time and counting up to that day.  Instead, the code below starts on 1 Jan 1900, increments by month to get to the start of some initial month and year, and continues incrementing the month up to the end of the final month and year, counting the times that the day of the week is some chosen value.

The `year` variables are the numerical year, the `month` variables are between $0-11$, and the `day` variables are between $0-6$, $0$ being Sunday.

In [22]:
%%time

ndays = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

def next_month(year: int, month: int, day: int) -> int:
    day = (day + ndays[month]) % 7
    if month == 1 and year % 4 == 0 and (year % 100 != 0 or year % 400 == 0):
        day = (day + 1) % 7
    return day
    
def p019(year_a: int, month_a: int, year_b: int, month_b: int, day: int) -> int:
    val = 0
        
    if (year_a, month_a) > (year_b, month_b):
        return 0
        
    if year_a < 1900:
        return 0

    # start on a Monday
    day = 1
    # increment up to `year_a`
    for year in range(1900, year_a):
        for month in range(12):
            day = next_month(year, month, day)
    # increment up to `month_a`
    for month in range(month_a):
        day = next_month(year_a, month, day)
    if year_a == year_b:
        # same year -> search between the two months
        for month in range(month_a, month_b + 1):
            if day == 0:
                val += 1
            day = next_month(year_a, month, day)
    else:
        # search in the remaining months of `year_a`
        for month in range(month_a, 12):
            if day == 0:
                val += 1
            day = next_month(year_a, month, day)
        # search the years between `year_a` and `year_b`
        for year in range(year_a + 1, year_b):
            for month in range(12):
                if day == 0:
                    val += 1
                day = next_month(year, month, day)
        # search the months of `year_b`
        for month in range(month_b + 1):
            if day == 0:
                val += 1
            day = next_month(year_b, month, day)

    return val


year_a = 1901
month_a = 0
year_b = 2000
month_b = 11
day = 0

ans = p019(year_a, month_a, year_b, month_b, day)
print(ans)

171
CPU times: user 415 μs, sys: 6 μs, total: 421 μs
Wall time: 409 μs


A different approach would be to consider that there are only $14$ possible calendars, one starting at each day of the week and is either a leap year or not.  The number of Sundays at the start of months can be added for each type of year, then the number of each calendar between the given years can be counted.